In [1]:
import random
import pandas as pd
import numpy as np
import os
import IPython.display as ipd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings(action='ignore') 

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
# 03-11-15:40에 추가된 것. 기존 베이스라인 아님
#building_info = pd.read_csv('data/building_info.csv')
# 2. '건물번호'를 기준으로 train과 test 데이터에 건물 정보 병합
#train = pd.merge(train, building_info, on='건물번호', how='left')
#test = pd.merge(test, building_info, on='건물번호', how='left')

In [4]:
# 03-11-15:20에 추가된 것. 기존 베이스라인 아님
# 1. 불쾌지수 (Discomfort Index, DI) 계산
# 전력 수요와 상관관계가 매우 높습니다.
#train['DI'] = 0.81 * train['기온(°C)'] + 0.01 * train['습도(%)'] * (0.99 * train['기온(°C)'] - 14.3) + 46.3
#test['DI'] = 0.81 * test['기온(°C)'] + 0.01 * test['습도(%)'] * (0.99 * test['기온(°C)'] - 14.3) + 46.3

# 2. 체감기온 (간이 공식)
# 기온과 습도를 조합한 파생 변수
#train['perceived_temp'] = train['기온(°C)'] + 0.33 * (train['습도(%)'] / 100 * 6.105 * np.exp(17.27 * train['기온(°C)'] / (237.7 + train['기온(°C)'])) ) - 4.0

In [5]:
# 1. 강수량, 일조, 일사는 0으로 채우기 (비 안 옴, 해 안 뜸)
train['강수량(mm)'] = train['강수량(mm)'].fillna(0)
train['일조(hr)'] = train['일조(hr)'].fillna(0)
train['일사(MJ/m2)'] = train['일사(MJ/m2)'].fillna(0)

# 2. 풍속, 습도는 앞뒤 데이터를 참고해서 자연스럽게 채우기
train['풍속(m/s)'] = train['풍속(m/s)'].interpolate(method='linear')
train['습도(%)'] = train['습도(%)'].interpolate(method='linear')

# 결과 확인
print(train.isnull().sum())

num_date_time    0
건물번호             0
일시               0
기온(C)            0
강수량(mm)          0
풍속(m/s)          0
습도(%)            0
일조(hr)           0
일사(MJ/m2)        0
전력소비량(kWh)       0
dtype: int64


In [6]:
for data in [train, test]:
    data['일시'] = pd.to_datetime(data['일시'])
    data['year'] = data['일시'].dt.year
    data['month'] = data['일시'].dt.month
    data['day'] = data['일시'].dt.day
    data['hour'] = data['일시'].dt.hour
    data['dayofweek'] = data['일시'].dt.dayofweek

In [7]:
# 6월 6일 평균 대비 6월 7일 평균 전력의 비율 계산
june_6 = train[train['일시'].dt.strftime('%m-%d') == '06-06'].groupby('건물번호')['전력소비량(kWh)'].mean()
june_7 = train[train['일시'].dt.strftime('%m-%d') == '06-07'].groupby('건물번호')['전력소비량(kWh)'].mean()

drop_ratio = (june_6 / june_7)
# 비율이 0.7 미만(30% 이상 급감)인 건물들만 보기
sensitive_buildings = drop_ratio[drop_ratio < 0.7].index.tolist()

print(f"공휴일에 민감한 건물 번호: {sensitive_buildings}")

공휴일에 민감한 건물 번호: [3, 17, 19, 20, 53, 54, 59, 60, 74, 77, 78, 80, 82, 83, 84]


In [8]:
train['is_holiday'] = train['일시'].dt.strftime('%m-%d').isin(['06-06'])

In [9]:
train['is_weekend'] = train['일시'].dt.dayofweek >= 5

In [10]:
# 불쾌지수 계산 함수 정의
def calculate_di(temp, humid):
    return 1.8 * temp - 0.55 * (1 - humid/100) * (1.8 * temp - 26) + 32

# 피처 생성
train['DI'] = calculate_di(train['기온(C)'], train['습도(%)'])
test['DI'] = calculate_di(test['기온(C)'], test['습도(%)'])

# 확인용
print(train[['기온(C)', '습도(%)', 'DI']].head())

   기온(C)  습도(%)        DI
0   18.6   42.0  63.09388
1   18.0   45.0  62.46400
2   17.7   45.0  62.08735
3   16.7   48.0  60.89884
4   18.4   43.0  62.88788


In [11]:
# [1] 공통 피처 리스트 정의 (순서가 중요합니다!)
features = [
    '건물번호', '기온(C)', '강수량(mm)', '풍속(m/s)', '습도(%)', 'DI', 
    'month', 'day', 'hour', 'dayofweek', 'is_holiday'
]

# [2] Test 데이터 전처리 (Train은 이미 되어 있다고 가정)
test['일시'] = pd.to_datetime(test['일시'])
test['month'] = test['일시'].dt.month
test['day'] = test['일시'].dt.day
test['hour'] = test['일시'].dt.hour
test['dayofweek'] = test['일시'].dt.weekday

# 공휴일 생성 (현충일, 광복절 + 주말)
test['is_holiday'] = test['일시'].dt.strftime('%m-%d').isin(['06-06', '08-15']).astype(int)
test['is_holiday'] = ((test['is_holiday'] == 1) | (test['dayofweek'] >= 5)).astype(int)

# DI(불쾌지수) 생성
test['DI'] = 1.8 * test['기온(C)'] - 0.55 * (1 - test['습도(%)']/100) * (1.8 * test['기온(C)'] - 26) + 32

# [3] 최종 학습/예측용 데이터셋 분리
X_train = train[features]
y_train = train['전력소비량(kWh)']
X_test = test[features]

print(f"✅ 데이터 준비 완료! (Train: {X_train.shape}, Test: {X_test.shape})")

# [4] XGBoost 모델 정의 및 학습
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

print("🚀 모델 학습 시작...")
model.fit(X_train, y_train)
print("✨ 학습 완료!")

# [5] 예측 수행
preds = model.predict(X_test)
print("📊 예측 완료! 상위 5개 결과:", preds[:5])

✅ 데이터 준비 완료! (Train: (204000, 11), Test: (16800, 11))
🚀 모델 학습 시작...
✨ 학습 완료!
📊 예측 완료! 상위 5개 결과: [1973.2194 1990.1342 1945.4261 1914.5276 1900.9559]


In [12]:
# [6] 제출용 파일 생성
submission = pd.read_csv('sample_submission.csv') # 파일명이 다를 수 있으니 확인!
submission['answer'] = preds

# 파일 저장
submission.to_csv('baseline_submission.csv', index=False)
print("🏁 제출 파일(baseline_submission.csv)이 생성되었습니다!")

🏁 제출 파일(baseline_submission.csv)이 생성되었습니다!
